# Workflow 2 – Ocean Microbial Signatures (OMS)

**Associated manuscript:** Huber et al. – *Ocean Microbiome Signatures capturing the spatial organization of metabolic potential in the global sunlit ocean* – *Nature Microbiology* (under review)

**Description:** This notebook covers the identification and characterization of Ocean Microbial Signatures (OMSs) using Non-negative Matrix Factorization (NMF):
- Loading and normalizing NMF output matrices (H and W)
- Geographic distribution of OMSs (major-contributing and monodominant)
- Functional characterization (PCA and Z-score analysis using KeyKOdb)
- Taxonomic characterization of OMS composition
- Environmental niche analysis

**Input files:** NMF matrices (`H_10.tsv`, `W_10.tsv`) and Supplementary Tables (S1, S3, S5, S8). Place them in the same directory as this notebook.

**Output files:** Key outputs include `OMS_samples_major_component.csv` (→ Table S11), `OMS_functionalAssignation_keyKO_OPUs_441817.csv` (→ Table S12), and taxonomic summary CSVs.

> **Note:** NMF decomposition was run externally using the R codes provided by Frioux et al 2023 (10.1016/j.chom.2023.05.024) . This notebook starts from the precomputed W and H matrices.

### 1. Load Required Packages

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import networkx as nx

import cartopy.crs as ccrs
import cartopy.feature as cfeature

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.transforms import offset_copy
from matplotlib.patches import Wedge
from matplotlib.patches import Wedge, Patch
import matplotlib.lines as lines
from matplotlib import rcParams

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import zscore


### 2. Load Input Tables

> **Input:** `TableS1_metadata.csv` – Metadata for 1,379 metagenomic samples (Table S1). Includes sample IDs, ocean basin, coordinates, and expedition information.

In [ ]:
# load metadata 
metadata_df = pd.read_csv("TableS1_metadata.csv", delimiter="\t")
metadata_df

### Load H table (as df)


> **Input:** `H_10.tsv` – NMF H matrix (10 OMSs × 1,379 samples). Each value represents the contribution of a given OMS to a sample. Corresponds to **Table S9**.

In [ ]:
H = pd.read_csv('TableS9_H-10_OMS_distribution_1379x10.csv', delimiter="\t") # 10 Rows x 1379 columns
H

In [ ]:
### Load W table (as df)


In [ ]:
W = pd.read_csv('TableS8_W_10_OMS_compostion_419797x10.csv', delimiter="\t") # 10 Rows x 1379 columns
W

#### Normalize H matrix

Column-wise normalization converts raw NMF contributions into **relative OMS abundances** per sample, allowing direct comparison across samples with different total contributions.

In [ ]:
def normalize_matrix(df):
    return df.div(df.sum(axis=0), axis=1)

# Normalizar la matriz H
df_normalized = normalize_matrix(df)

df_normalized

In [ ]:
#Merge df with metadata table (data)
#transpos the data
df_norm_t = df_normalized.T
# Add a new column 'MGnifyAssemblyNumb' using the index values of df_norm_t
df_norm_t['MGnifyAssemblyNumb'] = df_norm_t.index
df_norm_t

In [ ]:
#merge tables by MGnifyAssemblyNumb
df_norm_cat = pd.merge(metadata_df, df_norm_t, on='MGnifyAssemblyNumb', how='inner')  # Keep only matching rows
df_norm_cat
# Optionally, save the table to a CSV file
#df_norm_cat.to_csv('OMS_H_norm_to_map.csv', index=False)

In [ ]:
%matplotlib inline
# Define the colors for each group
colors = {
    "OMS_1": "#f37736",
    "OMS_2": "#0392cf",
    "OMS_3": "#7bc043",
    "OMS_4": "purple",
    "OMS_5": "#283655",
    "OMS_6": "#009688",
    "OMS_7": "#651e3e",
    "OMS_8": "#fed766",
    "OMS_9": "#2ab7ca",
    "OMS_10": "#b52b5e"
}

# Create the figure with 2 columns and 5 rows
fig, axes = plt.subplots(nrows=5, ncols=2, figsize=(15, 20), subplot_kw={'projection': ccrs.PlateCarree()})

# Flatten the axes array for easy iteration
axes = axes.flatten()

# Set a main title for the entire figure
plt.suptitle("Relative Contribution of Each OMS to the Samples", fontsize=16, y=0.92)

# Plot density for each OMS group in its respective subplot
for ax, (oms_group, color) in zip(axes, colors.items()):
    # Add map features
    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.add_feature(cfeature.LAKES, alpha=0.5)
    ax.add_feature(cfeature.RIVERS)

    # Set map extent (adjust according to your region of interest)
    ax.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())

    # Add latitude and longitude lines
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 10, 'color': 'gray'}
    gl.ylabel_style = {'size': 10, 'color': 'gray'}

    # Determine the size scaling factor
    size_scaling_factor = 100  # You can adjust this as needed
    sizes = df_norm_cat[oms_group] * size_scaling_factor

    # Get min and max sizes for the legend
    min_size = sizes.min()
    max_size = sizes.max()

    # Define size range for the legend based on min and max sizes
    size_labels = [min_size, min_size + (max_size - min_size) * 0.25, min_size + (max_size - min_size) * 0.5, max_size]
    density_values = [10, 25, 50, 100]  # Corresponding density values

    # Create proxy artists for the legend in each subplot
    legend_elements = [mlines.Line2D([], [], color='gray', marker='o', markersize=np.sqrt(size), linestyle='None', label=f'{density}') 
                       for size, density in zip(size_labels, density_values)]

    # Plot density for the current OMS group
    scatter = ax.scatter(
        df_norm_cat['decimalLongitude'], df_norm_cat['decimalLatitude'], 
        s=sizes, color=color, alpha=0.5, transform=ccrs.PlateCarree()
    )

    # Add legend to the current subplot
    ax.legend(handles=legend_elements, loc='lower right', fontsize=10, title='Density Value')

    # Set the title for the subplot
    ax.set_title(oms_group, fontsize=12)

# Adjust layout
plt.tight_layout(rect=[0, 0, 1, 0.9])

#save the plot
matplotlib.use('pdf')  # Cambiar temporalmente al backend PDF
plt.savefig("oms_maps_without_legend.pdf", dpi=300)
matplotlib.use('module://matplotlib_inline.backend_inline')  # Volver al backend interactivo

# Show the plot
plt.show()


#### Determine the number of OMSs needed to best describe a sample. 
To that purpose, we deﬁned an arbitrary cut-off of 90% accumulated relative abundance to describe all major OMS components of a sample. Thus, our operational deﬁnition posed that the set of OMS describing a sample are OMSs ordered by decreasing relative abundance, whose cumulated relative abundance is greater or equal to 0.9.   
Additionally, we describe as a ‘‘monodominant OMSs’’ an OMs whose relative abundance is greater than 0.9

In [ ]:
# Order OMSs per relative abundance and select  OMSs sign
def select_significant_ess(df_normalized, threshold=0.9):
    significant_ess = {}
    monodominant_ess = {}
    
    for sample in df_normalized.columns:
        sorted_df = df_normalized[sample].sort_values(ascending=False)
        cumsum = sorted_df.cumsum()
        significant_es = sorted_df[cumsum <= threshold].index.tolist()
        
        # Include additional OMSs if the sum does not reach the threshold
        if not significant_es or cumsum.iloc[len(significant_es) - 1] < threshold:
            additional_es = sorted_df.index[len(significant_es)]
            significant_es.append(additional_es)
            while cumsum.loc[additional_es] < threshold and len(significant_es) < len(sorted_df):
                additional_es = sorted_df.index[len(significant_es)]
                significant_es.append(additional_es)
        
        if cumsum.iloc[0] >= threshold:
            monodominant_ess[sample] = sorted_df.index[0]
        
        significant_ess[sample] = significant_es
    
    return significant_ess, monodominant_ess

# Select OMSs sig y OMSs monodominant
significant_ess, monodominant_ess = select_significant_ess(df_normalized)



# Count OMSs sig and monodominant
num_significant_ess = {sample: len(ess) for sample, ess in significant_ess.items()}
num_monodominant_ess = len(monodominant_ess)

In [ ]:
# Create a new DataFrame with the major-contributing OMSs
OMS_major_component = pd.DataFrame(index=df_normalized.columns)

for sample, ess in significant_ess.items():
    OMS_major_component.loc[sample, ess] = df_normalized.loc[ess, sample].values

# Rellenar los NaN con 0
OMS_major_component = OMS_major_component.fillna(0)

OMS_major_component

> Generates global map of OMS geographic distribution, colored by dominant OMS per sample. Corresponds to **Fig. 3a** (geographic distribution maps).

In [ ]:
OMS_major_component['SampleID'] = OMS_major_component.index
data_select = data[['SampleID', 'decimalLongitude', 'decimalLatitude']]
OMS_major_cat = pd.merge(OMS_major_component, data_select, on='SampleID', how='right')
OMS_major_cat

In [ ]:
# Load your dataframe (replace this with your actual dataframe)
df = OMS_major_cat.copy()

# Define latitude bins
latitude_bins = pd.cut(df['decimalLatitude'], bins=np.arange(-90, 100, 10), include_lowest=True)
df['Latitude_Bin'] = latitude_bins

# Define the specific order for OMS columns
oms_columns = ['OMS_1', 'OMS_2', 'OMS_3', 'OMS_4', 'OMS_5', 'OMS_6', 'OMS_7', 'OMS_8', 'OMS_9', 'OMS_10']

# Count active OMSs in each sample
df['Active_OMS_Count'] = (df[oms_columns] > 0).sum(axis=1)

# 1. Number of Samples with `n` Active OMSs in Each Latitude Range
samples_by_lat_bin = (
    df.groupby(['Latitude_Bin', 'Active_OMS_Count'])
    .size()
    .reset_index(name='Sample_Count')
)

# Pivot the data for a stacked bar plot
pivoted = samples_by_lat_bin.pivot(index='Latitude_Bin', columns='Active_OMS_Count', values='Sample_Count').fillna(0)

# Plot stacked bar chart
rcParams['pdf.fonttype'] = 42  # Ensures text is saved as text, not paths
rcParams['font.family'] = 'Helvetica'
plt.figure(figsize=(8, 4))
pivoted.plot(kind='bar', stacked=True, colormap='viridis', ax=plt.gca())
plt.xlabel('Latitude ', fontsize=9)
plt.ylabel('Number of Samples', fontsize=9)
plt.title('Number of Samples with significant OMSs Across Latitude', fontsize=10)
plt.xticks(rotation=45, fontsize=8)
plt.yticks(fontsize=8)
plt.legend(title="Active OMSs", loc="upper right")
plt.savefig('Number of Samples with significant OMSs Across Latitude.pdf', dpi=300, bbox_inches='tight')

# 2. OMS Better Represented at Each Latitude Rank
df['Latitude_Rank'] = df['decimalLatitude'].rank()

# Calculate average contribution of each OMS by latitude rank
oms_by_rank = df.groupby('Latitude_Bin')[oms_columns].mean().reset_index()

# Plot heatmap of OMS contributions across latitude rank
plt.figure(figsize=(8, 4))
sns.heatmap(
    oms_by_rank.set_index('Latitude_Bin').T,  # Transpose to have OMS on Y-axis
    cmap='viridis',
    cbar_kws={'label': 'Average Contribution'},
)
plt.title('OMS Contributions Across Latitude', fontsize=10)
plt.xlabel('Latitude Rank', fontsize=9)
plt.ylabel('OMS', fontsize=9)
plt.yticks(fontsize=8)
plt.xticks(rotation=45, fontsize=8)
plt.tight_layout()
# Save the plot to a file
plt.savefig('OMS Contributions Across Latitude.pdf', dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()


## 3. Functional Characterization of OMSs

To characterize the functional repertoire of each OMS, we cross-reference OPU annotations with the **KeyKOdb** marker gene database. This section:
1. Derives the best KO assignment per OPU from raw annotation strings
2. Flags OPUs that match a KeyKO marker gene
3. Builds the functional annotation table used for PCA and Z-score analyses

> Outputs feed into **Table S12** (functional contribution of OPUs per OMS).

In [ ]:
### 1.- Load tables
kko_df= pd.read_csv("TableS5_KeyKOdb.csv")
kko_df

ann = pd.read_csv("TableS3_OPUs_annotation.csv", delimiter=",")
ann

In [ ]:
#Merge files to add Key KO Information
ann_keyKO = ann.merge(kko_df, left_on='Key_KO', right_on='KO_id', how='left')
ann_keyKO
# Select rows where OPU_Type is "Unknown"
condition_unknown = ann_keyKO['OPU_Type'] == 'Unknown'

# Select rows where OPU_Type is "Known" and Key_KO is not "None"
condition_known_keyKO = (ann_keyKO['OPU_Type'] == 'Known') & (ann_keyKO['KO_id'].str.startswith('K0', na=False))

# Combine both conditions using the logical OR operator
combined_condition = condition_unknown | condition_known_keyKO

# Filter the table based on the combined condition
OPUs_KeyKO_UNK = ann_keyKO[combined_condition]

# Display the filtered table
OPUs_KeyKO_UNK

#### 3.2 PCA of Functional Categories Across OMSs

A PCA is run on the W matrix restricted to OPUs with KeyKO or Unknown annotation, to identify the metabolic functions most discriminant across OMSs.

**Preprocessing steps (as described in Methods):**
1. Randomly subsample Unknown OPUs to 15,000 rows (to avoid overrepresentation)
2. Exclude underrepresented functional categories (Lipid metabolism, Xenobiotics, etc.)
3. Z-score normalize contributions across OMSs

> Results reported in **Table S13** and **Fig. 4**.

In [ ]:
#Normalize W rows to compute the association strength of OPUs to each OMS
w_norm = w.div(w.sum(axis=1), axis=0)
# Add the index as a new column named 'OPU'
w_norm['OPU'] = w_norm.index

# Reset the index if you prefer it as a regular column
w_norm = w_norm.reset_index(drop=True)
w_norm


In [ ]:
### Merge w_norm  with OPUs_KeyKO_UNK

OPUs_KeyKO_UNK = OPUs_KeyKO_UNK.set_index('OPU')

# Replace NaN values in the 'Functional hierarchy 1' column with 'Unknown'
OPUs_KeyKO_UNK['Functional hierarchy 1'] = OPUs_KeyKO_UNK['Functional hierarchy 1'].fillna('Unknown')
OPUs_KeyKO_UNK['Functional hierarchy 2'] = OPUs_KeyKO_UNK['Functional hierarchy 2'].fillna('Unknown')

w_norm_ann = pd.merge(w_norm, OPUs_KeyKO_UNK, on='OPU', how='inner')  # Keep only matching rows
#w_ann = pd.merge(w, OPUs_KeyKO_UNK, on='OPU', how='inner')  # Keep only matching rows
w_norm_ann
#w_ann

In [ ]:
pca_df = w_ann[['OPU', 'OMS_1', 'OMS_2', 'OMS_3', 'OMS_4','OMS_5','OMS_6','OMS_7','OMS_8','OMS_9','OMS_10', "Functional hierarchy 1" ]]

#"Functional hierarchy 2"
pca_df
functions = pca_df['Functional hierarchy 1'].unique()
# Print uniques values
functions
pca_df

### As the functions are not equal represented, I have try this strategy before to run the PCA. The results were similar those obtained without filtering OPUs
1.- Randomly sample rows with Unknown in the column Functional hierarchy 1 and retain only 15,000 rows
2.- Remove also 
* Lipid metabolism                                 
* Biosynthesis of other secondary metabolites      
* Xenobiotics biodegradation                       
* Gene set  
which are less represented




In [ ]:
function_counts = pca_df['Functional hierarchy 1'].value_counts()
function_counts
#Randomly sample rows with Unknown in the column Functional hierarchy 1 and retain only 15,000 rows
#Remove also
#Lipid metabolism                                 732
#Biosynthesis of other secondary metabolites      206
#Xenobiotics biodegradation                       149
#Gene set
#which are less represented

# Filter rows with 'Unknown' in 'Functional hierarchy 1'
unknown_rows = pca_df[pca_df['Functional hierarchy 1'] == 'Unknown']

# Randomly sample 15,000 rows from the 'Unknown' subset
sampled_unknown = unknown_rows.sample(n=15000, random_state=42)

# Filter rows that do not have 'Unknown' in 'Functional hierarchy 1'
non_unknown_rows = pca_df[pca_df['Functional hierarchy 1'] != 'Unknown']

# Concatenate the sampled 'Unknown' rows with the non-'Unknown' rows
pca_filtered_df = pd.concat([non_unknown_rows, sampled_unknown])

# Shuffle the resulting dataframe to ensure randomization (optional)
pca_filtered_df = pca_filtered_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Define the categories to remove
categories_to_remove = [
    'Lipid metabolism',
    'Biosynthesis of other secondary metabolites',
    'Xenobiotics biodegradation',
    'Gene set'
]

# Remove rows matching the specified categories
pca_filtered_df = pca_filtered_df[~pca_filtered_df['Functional hierarchy 1'].isin(categories_to_remove)]

# Display the updated dataframe
pca_filtered_df


In [ ]:
function_counts = pca_filtered_df['Functional hierarchy 1'].value_counts()
function_counts

In [ ]:
melted_data = pd.melt(pca_filtered_df, id_vars=['Functional hierarchy 1','OPU'], value_vars=['OMS_1', 'OMS_2', 'OMS_3', 'OMS_4', 'OMS_5', 'OMS_6', 'OMS_7', 'OMS_8', 'OMS_9', 'OMS_10'], var_name='OMS_Name', value_name='Value')
melted_data

In [ ]:
data_wide = melted_data.pivot_table(index=['OPU', 'OMS_Name'], columns='Functional hierarchy 1', values='Value', fill_value=0).reset_index()
data_wide = data_wide.reset_index(drop=True)
# Drop rows with any missing values
data_wide = data_wide.dropna()

# Separate features and the target variable
features = data_wide.drop(['OMS_Name'], axis=1)
target = data_wide['OMS_Name']

In [ ]:
# Apply PCA + Tukey analysis
pca = PCA()
X_pca = pca.fit_transform(data_wide.drop('OMS_Name', axis=1))  # Drop 'OMS_Name' for PCA

# Compute the variance explained by each principal component
explained_variance = pca.explained_variance_ratio_

# Create a DataFrame for explained variance
explained_variance_df = pd.DataFrame({
    'Principal Component': [f'PC{i+1}' for i in range(len(explained_variance))],
    'Explained Variance': explained_variance
})

print(explained_variance_df)

# Get the loadings (eigenvectors)
loadings = pca.components_.T
loadings_df = pd.DataFrame(loadings, index=features.columns, columns=[f'PC{i+1}' for i in range(len(explained_variance))])

# Select the principal components that explain most of the variance
threshold = 0.95  # Adjust this threshold as needed
cumulative_variance = np.cumsum(explained_variance)
num_components = np.argmax(cumulative_variance >= threshold) + 1

print(f'Number of components explaining {threshold*100}% of the variance: {num_components}')

# Get the top contributing variables for the selected principal components
top_contributing_vars = loadings_df.iloc[:, :num_components]

print(top_contributing_vars)

# Extract top contributing variables
top_vars = top_contributing_vars.index.tolist()

# For each selected variable, perform Tukey's HSD test
for var in top_vars:
    # Create a DataFrame with the selected variable and the target variable
    var_df = pd.DataFrame({var: features[var], 'OMS_Name': target})
    
    # Perform Tukey's HSD test
    tukey_result = pairwise_tukeyhsd(endog=var_df[var], groups=var_df['OMS_Name'], alpha=0.05)
    
    print(f'Tukey\'s HSD test results for {var}:')
    print(tukey_result)

In [ ]:
top_contributing_vars

#### Z-Score Environmental information processing	

In [ ]:
#prepared the table
fw = w_ann[['OPU', 'OMS_1', 'OMS_2', 'OMS_3', 'OMS_4','OMS_5','OMS_6','OMS_7','OMS_8','OMS_9','OMS_10', "Functional hierarchy 1", "Functional hierarchy 2" ]]
en = fw[fw['Functional hierarchy 1'] == 'Environmental information processing']
en =en[['OPU', 'OMS_1', 'OMS_2', 'OMS_3', 'OMS_4','OMS_5','OMS_6','OMS_7','OMS_8','OMS_9','OMS_10', "Functional hierarchy 2" ]]
# Group data by 'Functional hierarchy 2'
grouped_data = en.groupby('Functional hierarchy 1')
en

In [ ]:
# Step 1: Compute the average contribution for each Functional hierarchy 2 and OMS
avg_contribution = en.iloc[:, 1:11].groupby(en['Functional hierarchy 1']).mean()
avg_contribution_zscore_en = avg_contribution.apply(zscore, axis=0)
# Display or save the z-scored results
avg_contribution_zscore_en

In [ ]:
# Ensure the OMS columns are ordered correctly
order = ['OMS_1', 'OMS_2', 'OMS_3', 'OMS_4', 'OMS_5', 'OMS_6', 'OMS_7', 'OMS_8', 'OMS_9', 'OMS_10']
avg_contribution_zscore_em = avg_contribution_zscore_em[order]
avg_contribution_zscore_en = avg_contribution_zscore_en[order]

# Set font and style configurations
rcParams['pdf.fonttype'] = 42  # Save text as text, not paths
rcParams['font.family'] = 'Helvetica'

# Create the figure and subplots
fig, axes = plt.subplots(2, 1, figsize=(10, 10), sharex=True, gridspec_kw={'height_ratios': [1, 1]})

# Plot the first heatmap (Energy Metabolisms Pathway)
sns.heatmap(avg_contribution_zscore_em, cmap="YlGnBu", annot=False, ax=axes[0])
#axes[0].set_title('Z-Scores of OMS Energy Metabolisms Pathway', fontsize=10)
axes[0].tick_params(axis='x', bottom=False, labelbottom=False)  # Hide x-axis ticks and labels
axes[0].set_ylabel('Z-Scores of OMS Energy Metabolisms Pathway', fontsize=10)

# Plot the second heatmap (Environmental Information Processing)
sns.heatmap(avg_contribution_zscore_en, cmap="YlGnBu", annot=False, ax=axes[1])
#axes[1].set_title('Z-Scores of OMS Environmental Information Processing', fontsize=16)
axes[1].set_xlabel('OMS', fontsize=14)
axes[1].set_xticklabels(order, rotation=45, ha='right')  # Rotate OMS labels for clarity
axes[1].set_ylabel('Z-Scores of OMS Environmental Information Processing', fontsize=10)

# Adjust layout and save
plt.tight_layout()
plt.savefig('OMS_Heatmaps_Comparison.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 5. Environmental Niche Characterization

Z-score analysis of **environmental variables** (temperature, salinity, nutrients, etc.) across OMSs, using mean values per OMS from the World Ocean Atlas 2018.

> **Input:** `environmenta_varables_mean_values.csv` (→ Table S10)

> Results correspond to the **canonical OMI (CANOMI) analysis** described in Methods and shown in **Supplementary Fig. S4**.

In [ ]:
env_var = pd.read_csv("environmenta_varables_mean_values.csv", index_col=0)
# Z-scores for each row (Functional hierarchy 2), along columns (OMS)
env_var_zscore = env_var.apply(zscore, axis=1)
# Display or save the z-scored results
env_var_zscore

In [ ]:
# Ensure the OMS columns are ordered correctly
order = ['OMS_1', 'OMS_2', 'OMS_3', 'OMS_4', 'OMS_5', 'OMS_6', 'OMS_7', 'OMS_8', 'OMS_9', 'OMS_10']
env_var_zscore = env_var_zscore[order]

rcParams['pdf.fonttype'] = 42  # Save text as text, not paths
rcParams['font.family'] = 'Helvetica'


# Plot the heatmap
plt.figure(figsize=(5, 5))
sns.heatmap(env_var_zscore, cmap="YlGnBu", annot=False)

# Set titles and labels
#plt.title('Z-Scores Environmental Variables', fontsize=16)
#plt.xlabel('OMS', fontsize=14)
#plt.ylabel('Functional Hierarchy 2', fontsize=14)

# Adjust layout and save
plt.tight_layout()
plt.savefig('Z-Scores Environmental Variables.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 6. Taxonomic Characterization of OMSs

Links OPU weights in the W matrix to their taxonomic classification (GTDB via CAT) to describe the bacterial and archaeal assemblages structuring each OMS.



In [ ]:
cat = pd.read_csv("TableS3_OPUs_Taxonomy_and_functional_assig_419979X19.xlsx",  delimiter=",")
cat

In [ ]:

# Normalize domain labels
domain_map = {
    'Bacteria': 'Bacteria',
    'Archaea': 'Archaea',
    'no hits to database': 'No hit',
    'no hit': 'No hit',
    'root': 'Root'
}
cat['Domain_clean'] = cat['domain'].map(domain_map).fillna(cat['domain'])

# Unique values in Domain
print(cat['domain'].unique())

In [ ]:
categories = ['Bacteria','Archaea','no hits to database','root']

# Base summary per domain (restricted/reordered to your categories)
summary = (cat.groupby('domain', dropna=False)
             .agg(n_OPUs=('OPU','nunique'),
                  Total_Contribution=('Total_Contribution','sum'),
                  mean_abundance=('Total_Contribution','mean'))
             .reindex(categories))


# Totals across the 4 categories
total_n = summary['n_OPUs'].sum()
total_abund = summary['Total_Contribution'].sum()

# Percent columns
summary['percent_OPUs'] = np.where(total_n > 0, (summary['n_OPUs'] / total_n) * 100, 0.0).round(2)
summary['percent_abundance'] = np.where(total_abund > 0, (summary['Total_Contribution'] / total_abund) * 100, 0.0).round(2)

# Append Total row
total_row = pd.Series({
    'n_OPUs': total_n,
    'total_abundance': total_abund,
    'mean_abundance': summary['mean_abundance'].mean(),  # or leave NaN if you prefer
    'percent_OPUs': 100.0 if total_n > 0 else 0.0,
    'percent_abundance': 100.0 if total_abund > 0 else 0.0
}, name='Total')

summary = pd.concat([summary, total_row.to_frame().T])

print(summary)